In [ ]:
from tqdm import tqdm
from astropy.io import fits
from astropy.table import Table

import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = "Times New Roman"

import torch
import torch.nn as nn
import torch.optim.lr_scheduler as lr_scheduler
import torch.optim as optim
import json

# DEVICE
# For 2 Process with different model and loss function, CUDA and MPS are available
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)

binsize = 5.0

In [ ]:
PATH_TO_DATA = "../lc-project2_version3/"
LCs = sorted(glob.glob(PATH_TO_DATA+"*.fits"))
LCs_filenames = [os.path.basename(x) for x in LCs]
PATH_TO_GROUND_TRUTH = "../ResponseFunction2/"
GTs = [PATH_TO_GROUND_TRUTH + "resp_"+"_".join(f.split(".")[0].split("_")[1:3])+".fits" for f in LCs_filenames]

def read_fits_file(file, bin=False, noisemode="GaussianNoise", snr=None):
    ''' 
    Reads a FITS file and returns a pandas DataFrame. The function also allows for adding Gaussian noise to the data based on a specified signal-to-noise ratio (SNR) and for binning the data into specified time intervals. The function takes the following parameters:
        - file: The path to the FITS file to be read.
        - bin: A boolean or a float/int value that determines whether to bin the data and the size of the bins (in days). If set to False, no binning is applied. If set to a float or int value, the data will be binned into intervals of that size (e.g., 0.5 for 0.5 TU, 1 for 1 TU, etc.).
        - noisemode: A string that specifies the type of noise to be added to the data. Currently, only "GaussianNoise" is supported, which adds Gaussian noise to the data based on the specified SNR.
        - snr: A float value that specifies the signal-to-noise ratio for adding Gaussian noise. If set to None or False, no noise will be added. If a float value is provided, Gaussian noise will be added to the data based on the SNR using the formula following Sacchi's lecture.
    '''
    fits_file = fits.open(file)
    df = Table(fits_file[1].data).to_pandas()
    df = df.interpolate(method='linear', axis=0)

    attr = df.keys()
    attr = attr[2:]
    if noisemode == "GaussianNoise": 

        if snr not in [None, False]:
            try:
                for k in attr:
                    '''
                    Add noise based on Sacchi's lecture
                    Noise contains Gaussian noise with \sigma = \sqrt{Es/(snr*En)}
                    '''
                    noise = np.random.randn(len(df))
                    Es = np.sum(df[k]**2)
                    En = np.sum(noise**2)
                    alpha = np.sqrt(Es/(snr*En))
                    df[k] = np.maximum(df[k] + alpha * noise, 0)
            except:
                raise Exception("SNR is not specified. Please specify SNR or set it to None.")

    if bin != False:
        if type(bin) in [float, int]:     
            df = binning(df,bin_size=float(bin))
        else:
            raise ValueError("Bin Size must be specified! and it must be float or int.")

    return df


def binning(df, bin_size):
    """
    Binning the time series data into bins of size bin_size (in days)
    i.e. 0.5 TU means 2 observations per TU
         1 TU means 1 observation per TU
         2 TU means 1 observation for every 2 TU
    """
    df['bin'] = df['time'] // bin_size
    #df = df.groupby('bin').agg({'lc_a': 'mean', 'lc_h': 'mean', 'lc_s': 'mean'})
    df = df.groupby('bin').mean()
    df['time'] = df.index * bin_size
    return df

def preprocessing(lightcurve,mode, snr=None):
    df = read_fits_file(lightcurve, snr=snr)
    if mode == 'binning':
        df = binning(df, binsize)
    elif mode == 'sampling':
        df = df[::binsize]
    elif mode =="nobin":
        return df
    else:
        raise ValueError("Mode not recognized")
    return df

def load_model(model_name,model,optimizer,scheduler):
    ''' Loads the model, optimizer, scheduler, epoch and loss history from a checkpoint file. 
    The checkpoint file is expected to be in the "Trained_Model" directory and named as "{model_name}.pt".  
    The checkpoint file should contain the following keys:
        - 'model_state_dict': The state dictionary of the model.
        - 'optimizer_state_dict': The state dictionary of the optimizer.
        - 'scheduler_state_dict': The state dictionary of the scheduler.
        - 'epoch': The epoch number at which the checkpoint was saved.
        - 'loss': The loss history up to the epoch at which the checkpoint was saved.
    '''
    checkpoint = torch.load(f"Trained_Model/{model_name}.pt")
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    epoch = checkpoint['epoch']
    loss_history = checkpoint['loss']
    return model, optimizer, scheduler, epoch, loss_history

S = []
true_driving_signal = []

snr = None
for i in [2,3,4]:
    df = preprocessing(LCs[i],mode='binning', snr=snr)
    S.append(df['lc_s'].to_numpy())
    true_driving_signal.append(df['lc_a'].to_numpy())

S = np.array(S)
true_driving_signal = np.array(true_driving_signal)
T = df['time'].to_numpy()

In [ ]:
def Cramers_Rule_Solver(S, R, P, b):
    '''
    Solves driving signal (a), Reverberation term and Propagation term using Cramer's rule for the system of linear equations represented by the matrices S, R, P and b. The function takes the following parameters:
        - S: A 3xN matrix representing the observed signals at three different X-ray frequencies.
        - R: Length 3 vector representing reverberating coefficients.
        - P: Length 3 vector representing the propagation coefficients.
        - b: Length 3 vector representing direct emission coefficients from AGN.
    Returns:
        - a: X-ray driving signal.
        - Reverberation_term: reverberation term.
        - Propagation_term: Propagation term
    '''
    S1, S2, S3 = S
    R1, R2, R3 = R
    P1, P2, P3 = P
    b1, b2, b3 = b

    numerator_1 = (R2 * P3 - R3 * P2) * S1 + (R3 * P1 - R1 * P3) * S2 + (R1 * P2 - R2 * P1) * S3
    numerator_2 = (S2 * P3 - S3 * P2) * b1 + (S3 * P1 - S1 * P3) * b2 + (S1 * P2 - S2 * P1) * b3
    numerator_3 = (R2 * S3 - R3 * S2) * b1 + (R3 * S1 - R1 * S3) * b2 + (R1 * S2 - R2 * S1) * b3

    denominator = np.linalg.det(np.array([b, R, P]).T )

    a = numerator_1 / denominator
    Reverberation_term = numerator_2/denominator
    Propagation_term = numerator_3/denominator

    tolerance = -1e-2
    
    if sum(a < tolerance) > 0.25 * len(a):
        raise ValueError("Combination of parameters not valid")  
    else:
        a = np.clip(a, a_min=0, a_max=None)

    return a, Reverberation_term, Propagation_term

In [ ]:
class REV_PROP_Model(nn.Module):
    def __init__(self, S):
        '''
        Initializes the REV_PROP_Model class, which is a PyTorch neural network module designed to model reverberation and propagation effects in X-ray signals. The class takes the following parameters:
            - S: A 3xN matrix representing the observed signals at three different X-ray frequencies.
        -----------------------------------------------------------------------------
        This class initializes driving signal (a), Reverberation term (Kappa), and Propagation term (Pi). RevKernel and PropKernel are respectively response function for X-ray reverberation and Accretion disk propagation. Both of them are learnable parameters with first guess as a uniform distribution with area under the curve as 1.
        '''
        super().__init__()

        assert S.shape[0] == 3
        self.N = S.shape[1]
        ### Dont' touch it ###
        self.R0 = torch.tensor([1.0, 1.5, 0.5])
        self.P0 = torch.tensor([0.5, 1.0, 1.0])
        self.b = torch.tensor( [1.0, 1.0, 1.0])
        #####

        self.R = torch.tensor([1.00, 1.00, 1.00]) * torch.clone(self.R0)
        self.P = (self.R0 + self.P0) - self.R

        self.RevKernel = nn.Parameter(torch.ones(self.N, dtype=torch.float32)/self.N)
        self.PropKernel = nn.Parameter(torch.ones(self.N,dtype=torch.float32)/self.N)

        self.a, self.Kappa_a, self.Pi_a = Cramers_Rule_Solver(S, self.R, self.P, self.b)     
    
    def forward(self):
        '''
            Computes the forward pass of the REV_PROP_Model, regenerating the Reverberation and Propagation terms based on the current estimates of the reverberation and propagation kernels.
            ---------------------
            return:
                - Reverberation: The reverberation term calculated by convolving the driving signal (a) with the reverberation kernel (RevKernel).
                - Propagation: The propagation term calculated by convolving the driving signal (a) with the propagation kernel (PropKernel).
        '''
        Reverberation   = torch.zeros(self.N, dtype=torch.float32)
        Propagation     = torch.zeros(self.N, dtype=torch.float32)

        for i in np.arange(self.N):
                K_ij = torch.flip(self.RevKernel[:i+1], [0])
                Pi_ij = torch.flip(self.PropKernel[:i+1], [0])
                a_j = self.a[:i+1]

                Reverberation[i] = torch.dot(K_ij, a_j)
                Propagation[i]   = torch.dot(Pi_ij, a_j)
        return Reverberation, Propagation

In [ ]:
S_tensor    = torch.tensor(S, dtype=torch.float32)

model = REV_PROP_Model(S_tensor)
Ka_tensor = model.Kappa_a
Pa_tensor = model.Pi_a
Ka_tensor = Ka_tensor.to(DEVICE)
Pa_tensor = Pa_tensor.to(DEVICE)

learning_rate = 1e-2
Loss = torch.nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

loss_history = []
resume = True
train = True
checkpoint_write = True

start_epoch = 0
epochs = 2000

model_name = f"Project_2_2Process_Cramers_{learning_rate:.0e}lr_{binsize}binsize_broadTH_varycombination_0"
scheduler = lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=1e-4, total_iters=0.6*epochs)

if train:
    if resume and os.path.exists(f"Trained_Model/{model_name}.pt"):
        model, optimizer, scheduler, start_epoch, loss_history = load_model(f"{model_name}", model, optimizer, scheduler)        
        scheduler.step()
        print(f"Resuming training from epoch {start_epoch}")
    else:
        os.makedirs("Trained_Model/", exist_ok=True)
        print("Starting training from scratch")

    model.train()

    for epoch in tqdm(range(start_epoch,epochs)):

        optimizer.zero_grad()

        Reverberation_term, Propagation_term = model()
        Reverberation_term = Reverberation_term.to(DEVICE)
        Propagation_term = Propagation_term.to(DEVICE)

        loss1 = Loss(Reverberation_term, Ka_tensor)
        loss1.backward()

        loss2 = Loss(Propagation_term, Pa_tensor)
        loss2.backward()
        
        # Optional: Clamp the parameters to be non-negative
        for p in model.parameters():
            p.data.clamp_(min = 0, max = None)

        optimizer.step()
        scheduler.step()
        loss_history.append(loss1.item() + loss2.item())

        if ((epoch+1) % 100.0 == 0) and checkpoint_write :
            torch.save({
                    'epoch': epoch +1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': loss_history,
                    },
                f"Trained_Model/{model_name}.pt")
            
else:
    model, optimizer, scheduler, start_epoch, loss_history = load_model(f"{model_name}",model,optimizer,scheduler)
    print("Load trained model")

In [ ]:
# Light Curve reconstruction

a = model.a
Reverberation_term, Propagation_term = model()
Kappa_a, Pi_a = Reverberation_term, Propagation_term

R_vec = model.R
R0_vec = model.R0
P_vec = model.P
b_vec = model.b

os.makedirs("Out_LCs_2P_Combi", exist_ok=True)
df_out = pd.DataFrame({"Time":T})

for index in range(3):
    lightcurve_out = b_vec[index] * a + R_vec[index] * Kappa_a + P_vec[index] * Pi_a
    df_out['lc_s_' + str(index)] = lightcurve_out.detach().numpy()

df_out.to_csv("Out_LCs_2P_Combi/" + model_name + "_lc.csv")

In [ ]:
# Write the training report -- Optimized response functions with normalization and parameters used in the training are also included in the report for future reference.

Prop_Response = model.PropKernel.cpu().detach().numpy()
Prop_Response /= np.trapz(Prop_Response,dx=binsize)

Rev_Response = model.RevKernel.cpu().detach().numpy()
Rev_Response /= np.trapz(Rev_Response,dx=binsize)

# Write training report
report= {
    "b" : model.b.numpy().tolist(),
    "R" : model.R.numpy().tolist(),
    "R0" : model.R0.numpy().tolist(),
    "P" : model.P.numpy().tolist(),
    "P0" : model.P0.numpy().tolist(),
    "Time": T.tolist(),
    "REV_Response" : Rev_Response.tolist(),
    "Prop_Response" : Prop_Response.tolist(),
    "snr" : snr,
    "learning_rate": learning_rate,
    "loss": loss_history,
    "n_iter" : len(loss_history)
}

os.makedirs('Training_log/', exist_ok=True)
with open(f'Training_log/{model_name}.json', "w") as outfile:
    json.dump(report, outfile)